# ♻️ Entrenamiento YOLO11**s** — TACO (mejora equilibrada)

Modelo **`yolo11s`** @ 640 px, 60 épocas. Guarda en Drive y es **reanudable**.
Mejora esperada: mAP 0.38 → ~0.42-0.44.

> ✅ Las celdas de evaluación y demo **cargan el `best.pt` explícitamente**, para
> no confundirlo nunca con el modelo base de fábrica (clases COCO).

## 1. Verificar la GPU

In [ ]:
!nvidia-smi

## 2. Instalar Ultralytics

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

## 3. Montar Drive y preparar el dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
from pathlib import Path

ZIP_EN_DRIVE = "/content/drive/MyDrive/TACO.v3-yolov5.yolo26.zip"
DESTINO = "/content/dataset_taco"

assert os.path.exists(ZIP_EN_DRIVE), f"No encuentro el ZIP en {ZIP_EN_DRIVE}."
os.makedirs(DESTINO, exist_ok=True)
with zipfile.ZipFile(ZIP_EN_DRIVE, "r") as z:
    z.extractall(DESTINO)

for split in ["train", "valid", "test"]:
    carpeta = Path(DESTINO) / split / "images"
    n = len(list(carpeta.glob("*.jpg"))) if carpeta.exists() else 0
    print(f"{split:6}: {n} imágenes")

### Ajustar `data.yaml` (lee las 18 clases automáticamente)

In [ ]:
import yaml
with open(f"{DESTINO}/data.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["train"] = f"{DESTINO}/train/images"
cfg["val"]   = f"{DESTINO}/valid/images"
cfg["test"]  = f"{DESTINO}/test/images"
cfg.pop("roboflow", None)
YAML_PATH = f"{DESTINO}/data.yaml"
with open(YAML_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
print("Nº de clases:", cfg["nc"]); print("Clases:", cfg["names"])

## 4. Entrenar YOLO11s @ 640 px (guardando en Drive)

In [ ]:
from ultralytics import YOLO

PROJECT_DIR = "/content/drive/MyDrive/taco_runs"
RUN_NAME = "residuos_taco_s"

model = YOLO("yolo11s.pt")
results = model.train(
    data=YAML_PATH,
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    project=PROJECT_DIR,
    name=RUN_NAME,
    save_period=10,
    plots=True,
)

## ▶️ ¿Se cortó por límite de GPU? Reanuda aquí
Ejecuta las celdas 1-3 y luego esta. Continúa desde el último checkpoint en Drive.

In [ ]:
from ultralytics import YOLO
LAST = "/content/drive/MyDrive/taco_runs/residuos_taco_s/weights/last.pt"
model = YOLO(LAST)
results = model.train(resume=True)

## 5. Evaluar el modelo
⚠️ Cargamos `best.pt` **explícitamente** para evaluar TU modelo (no el de fábrica).

In [ ]:
from ultralytics import YOLO

BEST = "/content/drive/MyDrive/taco_runs/residuos_taco_s/weights/best.pt"
model = YOLO(BEST)
print("Clases del modelo:", list(model.names.values()))   # debe ser Bottle, Can, ...

metrics = model.val(data="/content/dataset_taco/data.yaml")
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precisión    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

In [ ]:
import os
from IPython.display import Image as ColabImage, display
RUN_DIR = "/content/drive/MyDrive/taco_runs/residuos_taco_s"
for nombre in ["results.png", "confusion_matrix.png"]:
    ruta = os.path.join(RUN_DIR, nombre)
    if os.path.exists(ruta):
        print("==>", nombre); display(ColabImage(filename=ruta, width=820))

## 6. 🎯 Demo MULTI-OBJETO
⚠️ También cargamos `best.pt` explícitamente (clases de residuos, no COCO).

In [ ]:
from ultralytics import YOLO
import glob, random
import matplotlib.pyplot as plt

BEST = "/content/drive/MyDrive/taco_runs/residuos_taco_s/weights/best.pt"
model = YOLO(BEST)
print("Clases:", list(model.names.values()))   # verifica: Bottle, Can, Cigarette...

TEST_DIR = "/content/dataset_taco/test/images"
muestra = random.Random(7).sample(sorted(glob.glob(TEST_DIR + "/*.jpg")), 9)

fig, axes = plt.subplots(3, 3, figsize=(16, 16))
for ax, img_path in zip(axes.flat, muestra):
    r = model.predict(img_path, conf=0.20, augment=True, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1])
    ax.set_title(f"{len(r.boxes)} objetos detectados", fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/taco_runs/demo_s.png", dpi=110)
plt.show()
print("Guardado en tu Drive: taco_runs/demo_s.png")

## 7. Descargar el modelo

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/taco_runs/residuos_taco_s/weights/best.pt")